# Deep Learning for NLP - Intent Prediction




## Introduction


In this recitation, we will tackle two NLP tasks:
- "What's the purpose of a query?" (Intent Prediction)
- "What are the key words in a query?" (Slot Filing)

Intent classification focuses on predicting the intent of the query, while slot filling extracts semantic concepts in the query. For example the user query could be “Find me an action movie by Steven Spielberg”. The intent here is “find_movie” while the slots are “genre” with value “action” and “directed_by” with value “Steven Spielberg”.

Like we did in lecture, we will use the ATIS dataset which includes queries from people trying to find flight tickets between two destinations.

We will first solve the task of reading a raw query and detecting the **intent** of the query (e.g., find availability of flights, duration of flights, ground transportation services etc.)

Next, we will cover how to use a Transformer Encoder to read a raw query and detect **components** or **slots** in that query (which airports or cities, what days and time. This will be a review of the [Lecture 6 Colab code](https://colab.research.google.com/drive/1BKiZv__Ou_49RFYohAma_3Em52I1VrYO?usp=sharing) (since it wasn't covered in lecture).

## Data Preprocessing



Let's begin extracting the data from the ATIS dataset and turning into a form that we can use in our Deep Learning models.

The ATIS dataset is standard benchmark dataset widely used to build models for intent classification and slot filling tasks (we will explain all this shortly). You can find a very detailed explanation [here](https://catalog.ldc.upenn.edu/docs/LDC93S4B/corpus.html).

We will begin by loading the file and then partitioning into a test and a training set.

In [ ]:
import os
os.environ['KERAS_BACKEND'] = 'tensorflow'

import keras
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

keras.utils.set_random_seed(42)

In [ ]:
train_url = "https://www.dropbox.com/scl/fi/m2yj95tccxmzin3mnna47/atis_train_data.csv?rlkey=p61rpu2mwxjcb1ypfh89qzull&st=omw3dkpu&dl=1"
test_url = "https://www.dropbox.com/scl/fi/d1zwrv2jslo7j93p68l75/atis_test_data.csv?rlkey=0kl75tt54i2ccau9m1fxevlhn&st=w7n0orza&dl=1"

In [ ]:
df_train = pd.read_csv(train_url, index_col=0)
df_test = pd.read_csv(test_url, index_col=0)

In [ ]:
df_train.head()

,query,intent,slot filling
0,i want to fly from boston at 838 am and arriv...,flight,O O O O O B-fromloc.city_name O B-depart_time...
1,what flights are available from pittsburgh to...,flight,O O O O O B-fromloc.city_name O B-toloc.city_...
2,what is the arrival time in san francisco for...,flight_time,O O O B-flight_time I-flight_time O B-fromloc...
3,cheapest airfare from tacoma to orlando,airfare,B-cost_relative O O B-fromloc.city_name O B-t...
4,round trip fares from pittsburgh to philadelp...,airfare,B-round_trip I-round_trip O O B-fromloc.city_...


The first column of the Dataframe below contains the actual query that was asked. The second column indicates the intent (flight, flight time, etc), whereas the last column contains the slot filling structure. We will build a DNN to predict the "Intent" column from the incoming query.

We will first copy the data from Pandas dataframes to Numpy arrays.

In [ ]:
query_data_train = df_train['query'].values
intent_data_train = df_train['intent'].values

query_data_test = df_test['query'].values
intent_data_test = df_test['intent'].values

In [ ]:
max_query_length = 30

Let's first vectorize the input (i.e., the queries) with the STIE process.

In [ ]:
max_tokens = 10000  # This defines the max size of the vocabulary

text_vectorization = keras.layers.TextVectorization(
    max_tokens=max_tokens,
    output_mode="multi_hot",
    ngrams=2  # we will use bigrams
)
text_vectorization.adapt(query_data_train)


In [ ]:
text_vectorization.vocabulary_size()

6864

Turns out the vocab size even with bigrams is well short of 10,000.

In [ ]:
text_vectorization.get_vocabulary()[:20]

['[UNK]',
 np.str_('to'),
 np.str_('from'),
 np.str_('flights'),
 np.str_('the'),
 np.str_('flights from'),
 np.str_('on'),
 np.str_('what'),
 np.str_('me'),
 np.str_('flight'),
 np.str_('boston'),
 np.str_('show'),
 np.str_('san'),
 np.str_('i'),
 np.str_('denver'),
 np.str_('show me'),
 np.str_('a'),
 np.str_('san francisco'),
 np.str_('francisco'),
 np.str_('in')]

In [ ]:
input_vocab = text_vectorization.vocabulary_size()

query_data_train_vec = text_vectorization(query_data_train)
query_data_test_vec = text_vectorization(query_data_test)

Let's take a look at the vectorized data

In [ ]:
query_data_train_vec.shape

TensorShape([4978, 6864])

In [ ]:
query_data_train_vec[0]

<tf.Tensor: shape=(6864,), dtype=int64, numpy=array([0, 1, 1, ..., 0, 0, 0])>

Next, let's take a look at the output side i.e, the **intent** labels.

What are the different types of "intent" in the data?

In [ ]:
df_train['intent'].value_counts()

,count
intent,
flight,3666
airfare,423
ground_service,255
airline,157
abbreviation,147
aircraft,81
flight_time,54
quantity,51
flight+airfare,21


In [ ]:
# Output classes integer encoding
text_vectorization_intent = keras.layers.TextVectorization()
text_vectorization_intent.adapt(intent_data_train)

n_intents = text_vectorization_intent.vocabulary_size()


In [ ]:
n_intents

24

In [ ]:
text_vectorization_intent.get_vocabulary()

['',
 '[UNK]',
 np.str_('flight'),
 np.str_('airfare'),
 np.str_('groundservice'),
 np.str_('airline'),
 np.str_('abbreviation'),
 np.str_('aircraft'),
 np.str_('flighttime'),
 np.str_('quantity'),
 np.str_('flightairfare'),
 np.str_('distance'),
 np.str_('airport'),
 np.str_('city'),
 np.str_('groundfare'),
 np.str_('capacity'),
 np.str_('flightno'),
 np.str_('restriction'),
 np.str_('meal'),
 np.str_('airlineflightno'),
 np.str_('groundservicegroundfare'),
 np.str_('cheapest'),
 np.str_('airfareflighttime'),
 np.str_('aircraftflightflightno')]

In [ ]:
intent_data_train_sparse = text_vectorization_intent(intent_data_train)
intent_data_test_sparse = text_vectorization_intent(intent_data_test)

In [ ]:
intent_data_train_sparse.shape

TensorShape([4978, 1])

In [ ]:
intent_data_train_sparse[:5]

<tf.Tensor: shape=(5, 1), dtype=int64, numpy=
array([[2],
       [2],
       [8],
       [3],
       [3]])>

The above is our dependent categorical variable. Should we use `sparse_categorical_crossentropy` or `categorical_crossentropy`?

`sparse_categorical_crossentropy` because our dataset's labels are integers (rather than one-hot encoded vectors).


## Model

In [ ]:
# Intent Classification Model - a 24-way softmax!

units = 64

inputs = keras.layers.Input(shape=(input_vocab, ))
x = keras.layers.Dense(units=units, activation="relu")(inputs)
x = keras.layers.Dropout(0.5)(x)
outputs = keras.layers.Dense(n_intents, activation="softmax")(x)

model = keras.Model(inputs, outputs)
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 6864)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │       439,360 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 24)             │         1,560 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 440,920 (1.68 MB)

 Trainable params: 440,920 (1.68 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
model.compile(optimizer="adam",
              loss="sparse_categorical_crossentropy",
              metrics=["sparse_categorical_accuracy"])

In [ ]:
model.fit(x=query_data_train_vec,
          y=intent_data_train_sparse,
          epochs=20)

Epoch 1/20
156/156 ━━━━━━━━━━━━━━━━━━━━ 4s 10ms/step - loss: 1.9144 - sparse_categorical_accuracy: 0.6575
Epoch 2/20
156/156 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.5376 - sparse_categorical_accuracy: 0.8834
Epoch 3/20
156/156 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.2982 - sparse_categorical_accuracy: 0.9341
Epoch 4/20
156/156 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.1955 - sparse_categorical_accuracy: 0.9550
Epoch 5/20
156/156 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.1356 - sparse_categorical_accuracy: 0.9673
Epoch 6/20
156/156 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0932 - sparse_categorical_accuracy: 0.9829
Epoch 7/20
156/156 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0747 - sparse_categorical_accuracy: 0.9864
Epoch 8/20
156/156 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0591 - sparse_categorical_accuracy: 0.9896
Epoch 9/20
156/156 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.0490 - sparse_categorical_accuracy: 0.9899
Epoch 10/20
156/156 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step -

In [ ]:
model.evaluate(x=query_data_test_vec, y=intent_data_test_sparse)

28/28 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - loss: 0.4570 - sparse_categorical_accuracy: 0.9399


[0.41641101241111755, 0.9372900128364563]

94% accuracy! Is this impressive? Let's check what the baseline is.

In [ ]:
df_test['intent'].value_counts(normalize=True)

,proportion
intent,
flight,0.707727
airfare,0.053751
airline,0.042553
ground_service,0.040314
abbreviation,0.036954
capacity,0.023516
airport,0.020157
flight+airfare,0.013438
distance,0.011198


**Summary**

A "naive" model can get to 71% accuracy by classifying every test example as "flight" so the 94% accuracy of our intent model is a nice improvement!!



---



Next, we will switch to the [Lecture 6 Colab on the Transformer Encoder](https://colab.research.google.com/drive/1BKiZv__Ou_49RFYohAma_3Em52I1VrYO?usp=sharing) and review the code.